#### 01. Find the total matches played, total wins & total losses by each team

In [0]:
%sql
---- Creating new catalog, schema -----
use catalog sql_youtube_practise;
create schema if not exists pyspark;
use pyspark;
show current schema;

catalog,namespace
sql_youtube_practise,pyspark


In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
data = [
    ("India", "SL", "India"),
    ("SL", "Aus", "Aus"),
    ("SA", "Eng", "Eng"),
    ("Eng", "NZ", "NZ"),
    ("Aus", "India", "India")
]
columns = ["Team_1", "Team_2", "Winner"]
icc_world_cup_df = spark.createDataFrame(data, columns)
icc_world_cup_df.write.mode("overwrite").saveAsTable("icc_world_cup")
icc_world_cup_df.display()

Team_1,Team_2,Winner
India,SL,India
SL,Aus,Aus
SA,Eng,Eng
Eng,NZ,NZ
Aus,India,India


In [0]:
from pyspark.sql.functions import *
team1_df = icc_world_cup_df.select(
    col("Team_1").alias("team_name"),
    when(col("Winner") == col("Team_1"), 1).otherwise(0).alias("wins")
)
team2_df = icc_world_cup_df.select(
    col("Team_2").alias("team_name"),
    when(col("Winner") == col("Team_2"), 1).otherwise(0).alias("wins")
)
team = team1_df.unionAll(team2_df) \
        .groupBy("team_name") \
        .agg(
            count("*").alias("total_matches_played"),
            sum("Wins").alias("Wins")
        ) \
        .withColumn(
            "Losses",
            col("total_matches_played") - col("Wins")
        ) \
        .orderBy(col("Wins").desc())
team.display() 

team_name,total_matches_played,Wins,Losses
India,2,2,0
Eng,2,1,1
Aus,2,1,1
NZ,1,1,0
SL,2,0,2
SA,1,0,1


#### 02. Find the number of first-time customers and repeat customers for each order date.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
spark = SparkSession.builder.getOrCreate()
data = [
    (1, 100, "2022-01-01", 2000),
    (2, 200, "2022-01-01", 2500),
    (3, 300, "2022-01-01", 2100),
    (4, 100, "2022-01-02", 2000),
    (5, 400, "2022-01-02", 2200),
    (6, 500, "2022-01-02", 2700),
    (7, 100, "2022-01-03", 3000),
    (8, 400, "2022-01-03", 1000),
    (9, 600, "2022-01-03", 3000)
]
columns = ["order_id", "customer_id", "order_date", "order_amount"]
customer_orders_df = (
    spark.createDataFrame(data, columns)
         .withColumn("order_date", col("order_date").cast("date"))
)
customer_orders_df.write.mode("overwrite").saveAsTable("customer_orders")
customer_orders_df.display()

order_id,customer_id,order_date,order_amount
1,100,2022-01-01,2000
2,200,2022-01-01,2500
3,300,2022-01-01,2100
4,100,2022-01-02,2000
5,400,2022-01-02,2200
6,500,2022-01-02,2700
7,100,2022-01-03,3000
8,400,2022-01-03,1000
9,600,2022-01-03,3000


In [0]:
from pyspark.sql.functions import col, min, when, sum

# Step 1: First visit date of each customer
customer_first_visit = (
    customer_orders_df
    .groupBy("customer_id")
    .agg(
        min("order_date").alias("first_visit_date")
    )
)
customer_first_visit.display()

# Step 2: Join and create flags
customer_orders_new_df = (
    customer_orders_df
    .join(customer_first_visit, on="customer_id", how="inner")
    .withColumn(
        "first_visit_flag",
        when(col("order_date") == col("first_visit_date"), 1).otherwise(0)
    )
    .withColumn(
        "repeat_visit_flag",
        when(col("order_date") != col("first_visit_date"), 1).otherwise(0)
    )
)
customer_orders_new_df.display()

# Step 3: Aggregate
final_df = (
    customer_orders_new_df
    .groupBy("order_date")
    .agg(
        sum("first_visit_flag").alias("first_visits"),
        sum("repeat_visit_flag").alias("repeat_visits")
    )
    .orderBy("order_date")
)
final_df.display()

customer_id,first_visit_date
100,2022-01-01
200,2022-01-01
300,2022-01-01
400,2022-01-02
500,2022-01-02
600,2022-01-03


customer_id,order_id,order_date,order_amount,first_visit_date,first_visit_flag,repeat_visit_flag
100,7,2022-01-03,3000,2022-01-01,0,1
200,2,2022-01-01,2500,2022-01-01,1,0
300,3,2022-01-01,2100,2022-01-01,1,0
400,8,2022-01-03,1000,2022-01-02,0,1
500,6,2022-01-02,2700,2022-01-02,1,0
600,9,2022-01-03,3000,2022-01-03,1,0
100,4,2022-01-02,2000,2022-01-01,0,1
400,5,2022-01-02,2200,2022-01-02,1,0
100,1,2022-01-01,2000,2022-01-01,1,0


order_date,first_visits,repeat_visits
2022-01-01,3,0
2022-01-02,2,1
2022-01-03,1,2


###### Using windows fucntion

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

window_spec = Window.partitionBy("customer_id")

final_df = (
    customer_orders_df
    .withColumn(
        "first_visit_date",
        min("order_date").over(window_spec)
    )
    .withColumn(
        "first_visit_flag",
        when(col("order_date") == col("first_visit_date"), 1).otherwise(0)
    )
    .withColumn(
        "repeat_visit_flag",
        when(col("order_date") != col("first_visit_date"), 1).otherwise(0)
    )
    .groupBy("order_date")
    .agg(
        sum("first_visit_flag").alias("first_visits"),
        sum("repeat_visit_flag").alias("repeat_visits")
    )
    .orderBy("order_date")
)
final_df.display()

order_date,first_visits,repeat_visits
2022-01-01,3,0
2022-01-02,2,1
2022-01-03,1,2


#### 03. Find each employee's 
- most visited floor, 
- total number of visits, and
- distinct resources they used.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
spark = SparkSession.builder.getOrCreate()
data = [
    ("A", "Bangalore", "A@gmail.com", 1, "CPU"),
    ("A", "Bangalore", "A1@gmail.com", 1, "CPU"),
    ("A", "Bangalore", "A2@gmail.com", 2, "DESKTOP"),
    ("B", "Bangalore", "B@gmail.com", 2, "DESKTOP"),
    ("B", "Bangalore", "B1@gmail.com", 2, "DESKTOP"),
    ("B", "Bangalore", "B2@gmail.com", 1, "MONITOR")
]
columns = ["name", "address", "email", "floor", "resources"]
entries_df = spark.createDataFrame(data, columns)
entries_df.write.mode("overwrite").saveAsTable("entries")
entries_df.display()

name,address,email,floor,resources
A,Bangalore,A@gmail.com,1,CPU
A,Bangalore,A1@gmail.com,1,CPU
A,Bangalore,A2@gmail.com,2,DESKTOP
B,Bangalore,B@gmail.com,2,DESKTOP
B,Bangalore,B1@gmail.com,2,DESKTOP
B,Bangalore,B2@gmail.com,1,MONITOR


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

part1 = entries_df.groupBy(
    "name"
    ).agg(
        count(col("floor")).alias("total_visits"),
        concat_ws(",", collect_set(col("resources"))).alias("resources_used")
    )
part1.display()

part2 = entries_df.groupBy(
    col("name"),col("floor")
).agg(
    count(col("floor")).alias("no_of_floor_visit")
).withColumn(
    "rank",
    dense_rank().over(Window.partitionBy(col("name")).orderBy(col("no_of_floor_visit").desc()))
)
part2.display()

final_df = part2.join(
    part1,
    part2["name"] == part1["name"],
    "inner"
).filter(
    col("rank") == 1
).select(
    part1["name"],
    part1["total_visits"],
    part2["floor"].alias("most_visited_floor"),
    part1["resources_used"]
).orderBy(col("name"))
final_df.display()

name,total_visits,resources_used
A,3,"CPU,DESKTOP"
B,3,"DESKTOP,MONITOR"


name,floor,no_of_floor_visit,rank
A,1,2,1
A,2,1,2
B,2,2,1
B,1,1,2


name,total_visits,most_visited_floor,resources_used
A,3,1,"CPU,DESKTOP"
B,3,2,"DESKTOP,MONITOR"


##### 04. Find the person IDs and names of all users whose friends' total score is greater than 100, along with the number of friends and their total score.

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
person_data = [
    (1, "Alice", "alice2018@hotmail.com", 88),
    (2, "Bob", "bob2018@hotmail.com", 11),
    (3, "Davis", "davis2018@hotmail.com", 27),
    (4, "Tara", "tara2018@hotmail.com", 45),
    (5, "John", "john2018@hotmail.com", 63)
]
person_columns = ["PersonID", "Name", "Email", "Score"]
person_df = spark.createDataFrame(person_data, person_columns)
person_df.write.mode("Overwrite").saveAsTable("person")

friend_data = [
    (1, 2),
    (1, 3),
    (2, 1),
    (2, 3),
    (3, 5),
    (4, 2),
    (4, 3),
    (4, 5)
]
friend_columns = ["PersonID", "FriendID"]
friend_df = spark.createDataFrame(friend_data, friend_columns)
friend_df.write.mode("Overwrite").saveAsTable("friend")

friend_df.display()
person_df.display()

PersonID,FriendID
1,2
1,3
2,1
2,3
3,5
4,2
4,3
4,5


PersonID,Name,Email,Score
1,Alice,alice2018@hotmail.com,88
2,Bob,bob2018@hotmail.com,11
3,Davis,davis2018@hotmail.com,27
4,Tara,tara2018@hotmail.com,45
5,John,john2018@hotmail.com,63


In [0]:
from pyspark.sql.functions import col, asc, count, sum

a = friend_df.alias("f").join(
    person_df.alias("p"),
    col("f.FriendID") == col("p.PersonID"),
    "inner"
).groupBy(
    col("f.PersonID")
).agg(
    count(col("f.FriendID")).alias("total_friends"),
    sum(col("p.Score")).alias("total_score")
).filter(
    col("total_score") > 100
)

final_df = a.join(
    person_df.alias("p1"),
    a["PersonID"] == col("p1.PersonID"),
    "inner"
).select(
    a["PersonID"],
    col("p1.Name"),
    a["total_friends"],
    a["total_score"]
)

a.display()
final_df.display()

PersonID,total_friends,total_score
2,2,115
4,3,101


PersonID,Name,total_friends,total_score
2,Bob,2,115
4,Tara,3,101
